In [1]:
import sqlite3
import re
import pandas as pd
import os

from google.colab import files

uploaded = files.upload()

Saving pockettoons.db to pockettoons.db


In [2]:
import os

print(os.listdir())

['.config', 'pockettoons.db', 'sample_data']


In [3]:
!pip install groq -q
print("✅ Groq installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 1.4 MB/s eta 0:00:00
✅ Groq installed!


In [5]:
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

Enter Groq API Key: ··········


In [6]:
from groq import Groq
client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

print("✅ New client connected!")

✅ New client connected!


In [8]:
import sqlite3, pandas as pd

DB_PATH = 'pockettoons.db'

if 'conn' in locals() and conn:
    conn.close()
conn = sqlite3.connect(DB_PATH)
for table in ["users","sessions","transactions","content_views"]:
    count = pd.read_sql(f"SELECT COUNT(*) as rows FROM {table}", conn).iloc[0,0]
    print(f"{table}: {count} rows")
conn.close()

users: 10000 rows
sessions: 50000 rows
transactions: 28000 rows
content_views: 30000 rows


## **Defining Schema**

In [10]:
schema = """
You are a data analyst for PocketToons, a comic/webtoon streaming app.
You have access to a SQLite database with these tables:

TABLE users
  user_id      INTEGER  -- unique user ID
  country      TEXT     -- 'IN','US','GB','AU','CA','DE','SG','AE','OTHER'
  platform     TEXT     -- 'android','ios','web'
  plan_type    TEXT     -- 'free','basic','premium'
  signup_date  TEXT     -- 'YYYY-MM-DD'
  age_bucket   TEXT     -- '13-17','18-24','25-34','35-44','45+'
  gender       TEXT     -- 'M','F','other'
  is_active    INTEGER  -- 1=active, 0=churned

TABLE sessions
  session_id        INTEGER
  user_id           INTEGER
  platform          TEXT
  session_start     TEXT    -- ISO datetime, use DATE(session_start) for date
  session_end       TEXT
  duration_seconds  INTEGER
  country           TEXT

TABLE transactions
  transaction_id  INTEGER
  user_id         INTEGER
  timestamp       TEXT
  amount_usd      REAL
  plan_type       TEXT
  status          TEXT    -- 'success','failed','refunded'
  payment_method  TEXT    -- 'card','upi','wallet','bank_transfer'
  country         TEXT

TABLE content_views
  view_id          INTEGER
  user_id          INTEGER
  content_id       INTEGER
  content_title    TEXT
  genre            TEXT    -- 'romance','action','thriller','comedy','horror','sci-fi','slice_of_life'
  episode_number   INTEGER
  viewed_at        TEXT
  completion_pct   REAL    -- 0.0 to 1.0
  is_ppv           INTEGER -- 1=pay-per-view
  country          TEXT
  platform         TEXT

IMPORTANT RULES:
- DAU = COUNT(DISTINCT user_id) per DATE(session_start) from sessions
- MAU = COUNT(DISTINCT user_id) per month from sessions
- D7 retention = users who had a session within 7 days of their signup_date
- Revenue = SUM(amount_usd) WHERE status = 'success'
- Data date range: 2024-10-01 to 2025-01-31
- NEVER add WHERE clause to filter date range unless user specifically asks for it
- Always COUNT all available data unless filtered by user
- For D7 retention: count sessions BETWEEN signup_date AND DATE(signup_date, '+7 days') inclusive
- Always start retention window FROM signup_date itself, not +1 day
"""

print(" Schema defined!")

 Schema defined!


In [11]:

MODEL="llama-3.3-70b-versatile"
print(f" Using model: {MODEL}")

 Using model: llama-3.3-70b-versatile


In [12]:
def is_ambiguous(question):
    """
    Ask Groq: is this question too vague to answer?
    Returns True if we need to ask for clarification
    """
    prompt = f"""
    A user asked this analytics question: "{question}"

    Is this question too vague to write a specific SQL query?
    For example:
    - "How are we doing?" → TOO VAGUE (what metric?)
    - "How many users do we have?" → TOO VAGUE (active? total? paying?)
    - "What was DAU last week?" → CLEAR ENOUGH
    - "Revenue for US users in January?" → CLEAR ENOUGH

    Reply with ONLY one word: AMBIGUOUS or CLEAR
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    result = response.choices[0].message.content.strip().upper()
    return "AMBIGUOUS" in result



In [13]:
def generate_sql(question):
    """
    Takes a natural language question
    Returns SQL query to answer it
    """
    prompt = f"""
    {schema}

    Write a SQLite SQL query to answer this question:
    "{question}"

    RULES:
    - Return ONLY the SQL query, nothing else
    - No explanations, no markdown, no backticks
    - Use proper SQLite syntax
    - Always use readable column aliases
    - Limit to 20 rows unless asked for more
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    sql = response.choices[0].message.content.strip()
    sql = re.sub(r"```sql|```", "", sql).strip()
    return sql

print("SQL generator ready!")

SQL generator ready!


In [14]:
def run_sql(sql):
    """
    Runs SQL on our database
    Returns (columns, rows) or error message
    """
    try:
        conn   = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(sql)
        cols = [d[0] for d in cursor.description]
        rows = cursor.fetchall()
        conn.close()
        return cols, rows, None
    except Exception as e:
        return None, None, str(e)

print("SQL runner ready!")

SQL runner ready!


In [15]:
def generate_answer(question, cols, rows):
    """
    Takes SQL results
    Returns a plain English answer
    """

    result_text = " | ".join(cols) + "\n"
    result_text += "\n".join([str(row) for row in rows[:10]])

    prompt = f"""
    The user asked: "{question}"

    The SQL query returned this data:
    {result_text}

    Write a clear, friendly 2-3 sentence answer in plain English.
    Include the key numbers from the data.
    No SQL, no technical jargon, just a clean answer.
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip()

print("Answer generator ready!")

Answer generator ready!


In [16]:
def ask(question):
    """
    MAIN FUNCTION
    Takes any question → returns answer
    """
    print(f"\n{'='*60}")
    print(f"❓ Question: {question}")
    print(f"{'='*60}")


    if is_ambiguous(question):
        print("This question is ambiguous!")
        print(" Please clarify — for example:")
        print("   → Are you asking about total users or active users?")
        print("   → Which time period?")
        print("   → Which country or platform?")
        return


    sql = generate_sql(question)
    print(f"\n Generated SQL:\n{sql}")


    cols, rows, error = run_sql(sql)

    if error:
        print(f"\n SQL Error: {error}")
        print("Try rephrasing your question!")
        return


    df = pd.DataFrame(rows, columns=cols)
    print(f"\n📊 Raw Results:")
    print(df.to_string(index=False))


    answer = generate_answer(question, cols, rows)
    print(f"\nAnswer: {answer}")

print("ask() function ready!")
print("\n Agent is ready! Use ask('your question here')")


ask() function ready!

 Agent is ready! Use ask('your question here')


## **Testing on Agent using 5 Types of Questions.**

### **Type 1. Simple Aggregate**

In [17]:
ask("What was our DAU on January 15, 2025?")



❓ Question: What was our DAU on January 15, 2025?

 Generated SQL:
SELECT COUNT(DISTINCT s.user_id) AS daily_active_users 
FROM sessions s 
WHERE DATE(s.session_start) = '2025-01-15'

📊 Raw Results:
 daily_active_users
                384

Answer: On January 15, 2025, we had 384 daily active users. This means that 384 people used our service on that specific day. Our daily active user count for January 15, 2025, was 384.


In [18]:
ask("What was our MAU for each month?")


❓ Question: What was our MAU for each month?

 Generated SQL:
SELECT 
  STRFTIME('%Y-%m', session_start) AS month,
  COUNT(DISTINCT user_id) AS monthly_active_users
FROM 
  sessions
GROUP BY 
  STRFTIME('%Y-%m', session_start)
ORDER BY 
  month
LIMIT 20

📊 Raw Results:
  month  monthly_active_users
2024-10                  7094
2024-11                  7073
2024-12                  7177
2025-01                  7153

Answer: Our monthly active users (MAU) for the past few months were: 7,094 in October 2024, 7,073 in November 2024, 7,177 in December 2024, and 7,153 in January 2025. These numbers show a relatively stable trend, with a slight increase in December. Overall, our MAU has remained around 7,100 users per month.


### **Type 2: Filtered Aggregate**

In [19]:
ask("What is the total revenue from US users?")



❓ Question: What is the total revenue from US users?

 Generated SQL:
SELECT SUM(t.amount_usd) AS total_revenue 
FROM transactions t 
JOIN users u ON t.user_id = u.user_id 
WHERE u.country = 'US' AND t.status = 'success'

📊 Raw Results:
 total_revenue
      35844.87

Answer: The total revenue from US users is approximately $35,844.87. This is the total amount generated from users in the United States. This revenue figure gives us a snapshot of the earnings from this specific user group.


In [20]:
ask("How many premium users do we have in India?")



❓ Question: How many premium users do we have in India?

 Generated SQL:
SELECT COUNT(DISTINCT user_id) AS premium_user_count 
FROM users 
WHERE country = 'IN' AND plan_type = 'premium'

📊 Raw Results:
 premium_user_count
                886

Answer: We have 886 premium users in India. This number gives us a sense of our user base in the country and can help us understand how our services are being utilized. With nearly 900 premium users, we have a solid foundation to build on and continue growing our presence in India.


### **Type 3: Cohort / Retention**


In [21]:
ask("What is the D7 retention rate for each signup cohort?")


❓ Question: What is the D7 retention rate for each signup cohort?

 Generated SQL:
SELECT 
  u.signup_date AS cohort,
  COUNT(DISTINCT CASE WHEN s.session_start BETWEEN u.signup_date AND DATE(u.signup_date, '+7 days') THEN s.user_id END) AS retained_users,
  COUNT(DISTINCT u.user_id) AS total_users,
  ROUND(COUNT(DISTINCT CASE WHEN s.session_start BETWEEN u.signup_date AND DATE(u.signup_date, '+7 days') THEN s.user_id END) * 1.0 / COUNT(DISTINCT u.user_id), 2) AS d7_retention_rate
FROM 
  users u
  LEFT JOIN sessions s ON u.user_id = s.user_id
GROUP BY 
  u.signup_date
ORDER BY 
  u.signup_date
LIMIT 20

📊 Raw Results:
    cohort  retained_users  total_users  d7_retention_rate
2024-10-01               4           35               0.11
2024-10-02              19           87               0.22
2024-10-03              22           82               0.27
2024-10-04              22           81               0.27
2024-10-05              18           74               0.24
2024-10-06        

In [22]:
ask("Do premium users have better D7 retention than free users?")



❓ Question: Do premium users have better D7 retention than free users?

 Generated SQL:
SELECT 
  u.plan_type AS plan,
  COUNT(DISTINCT CASE WHEN s.session_start BETWEEN u.signup_date AND DATE(u.signup_date, '+7 days') THEN u.user_id END) AS retained_users,
  COUNT(DISTINCT u.user_id) AS total_users,
  ROUND(COUNT(DISTINCT CASE WHEN s.session_start BETWEEN u.signup_date AND DATE(u.signup_date, '+7 days') THEN u.user_id END) * 1.0 / COUNT(DISTINCT u.user_id), 4) AS d7_retention_rate
FROM 
  users u
  LEFT JOIN sessions s ON u.user_id = s.user_id
WHERE 
  u.plan_type IN ('free', 'premium')
GROUP BY 
  u.plan_type
LIMIT 20

📊 Raw Results:
   plan  retained_users  total_users  d7_retention_rate
   free            1252         5451             0.2297
premium             457         2009             0.2275

Answer: Premium users have a similar D7 retention rate to free users, with 22.75% of premium users retained compared to 22.97% of free users. This means that out of 2,009 premium users, 

### **Type 4: Comparison**


In [23]:
ask("Compare total sessions between January 2025 and December 2024")


❓ Question: Compare total sessions between January 2025 and December 2024

 Generated SQL:
SELECT 
  STRFTIME('%Y', session_start) AS year,
  COUNT(DISTINCT session_id) AS total_sessions
FROM 
  sessions
GROUP BY 
  STRFTIME('%Y', session_start)
ORDER BY 
  year DESC
LIMIT 20

📊 Raw Results:
year  total_sessions
2025           12624
2024           37376

Answer: In January 2025, there were 12,624 total sessions, which is significantly fewer than the 37,376 sessions in December 2024. This represents a substantial decrease in sessions from the end of 2024 to the beginning of 2025. Overall, December 2024 had about 3 times as many sessions as January 2025.


In [24]:
ask("Compare views by genre between October 2024 and November 2024 only")




❓ Question: Compare views by genre between October 2024 and November 2024 only

 Generated SQL:
SELECT 
  genre AS Genre, 
  SUM(CASE WHEN STRFTIME('%Y-%m', viewed_at) = '2024-10' THEN 1 ELSE 0 END) AS October_2024_Views, 
  SUM(CASE WHEN STRFTIME('%Y-%m', viewed_at) = '2024-11' THEN 1 ELSE 0 END) AS November_2024_Views
FROM 
  content_views
WHERE 
  viewed_at BETWEEN '2024-10-01' AND '2024-11-30'
GROUP BY 
  genre
ORDER BY 
  October_2024_Views DESC
LIMIT 20

📊 Raw Results:
        Genre  October_2024_Views  November_2024_Views
      romance                1204                 1045
     thriller                1200                 1063
       comedy                1157                 1028
       horror                1126                 1070
       sci-fi                1112                 1027
       action                1088                 1029
slice_of_life                 733                  728

Answer: Between October and November 2024, views decreased across most genres,

### **Type 5: Ambiguous — agent should ask for clarification**


In [25]:
ask("How are we doing?")


❓ Question: How are we doing?
This question is ambiguous!
 Please clarify — for example:
   → Are you asking about total users or active users?
   → Which time period?
   → Which country or platform?


In [26]:
ask("Show me users")



❓ Question: Show me users
This question is ambiguous!
 Please clarify — for example:
   → Are you asking about total users or active users?
   → Which time period?
   → Which country or platform?
